# Skin Cancer (ISIC, 9 classes) — Transfer Learning Benchmark
Fills three tables:
1. **Table 1** — Accuracy/Precision/Recall/F1/AUC for 8 transfer-learning models
2. **Table 2** — Classical classifiers trained on deep features
3. **Table 3** — Computational efficiency (params, size, FLOPs, inference time)

**Before running:** Runtime → Change runtime type → GPU.

## 1. Install dependencies

In [ ]:
!pip install -q kaggle torchmetrics xgboost thop scikit-learn --upgrade

## 2. Download the dataset (one-time Colab Secrets setup, then fully automatic)

**One-time setup (30 seconds):**
1. Get a Kaggle API key: https://www.kaggle.com/settings → API → "Create New Token" (downloads `kaggle.json` — open it in a text editor to see your username & key)
2. In Colab, click the 🔑 **key icon (Secrets)** in the left sidebar
3. Add secret `KAGGLE_USERNAME` → your Kaggle username
4. Add secret `KAGGLE_KEY` → your Kaggle API key
5. Toggle **Notebook access** on for both

After that, this cell runs with **zero prompts**, every time.

In [ ]:
import os, glob
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

os.makedirs("/content/data", exist_ok=True)
!kaggle datasets download -d nodoubttome/skin-cancer9-classesisic -p /content/data --unzip

# The dataset unzips to something like:
# /content/data/Skin cancer ISIC The International Skin Imaging Collaboration/Train
# /content/data/Skin cancer ISIC The International Skin Imaging Collaboration/Test
# Adjust DATA_ROOT below if the folder name differs after unzip (print it to check).
print(glob.glob("/content/data/*"))

## 3. Config

In [ ]:
DATA_ROOT = "/content/data/Skin cancer ISIC The International Skin Imaging Collaboration"
TRAIN_DIR = os.path.join(DATA_ROOT, "Train")
TEST_DIR  = os.path.join(DATA_ROOT, "Test")

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 10          # bump up for real results; kept low so it's Colab-friendly
LR         = 1e-4
DEVICE     = "cuda" if __import__("torch").cuda.is_available() else "cpu"
NUM_CLASSES = 9

print("Device:", DEVICE)

## 4. Data loaders

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

full_train = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
test_ds    = datasets.ImageFolder(TEST_DIR,  transform=eval_tf)

# split a validation set out of train (85/15)
val_size = int(0.15 * len(full_train))
train_size = len(full_train) - val_size
train_ds, val_ds = random_split(full_train, [train_size, val_size])
# validation set should use eval transforms, not train-time augmentation
val_ds.dataset.transform = eval_tf

class_names = full_train.classes
print("Classes:", class_names)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 5. Model zoo (matches Table 1's row list)

In [ ]:
import torchvision.models as tvm

def build_model(name, num_classes=NUM_CLASSES):
    """Returns a torchvision model with its classifier head replaced."""
    if name == "AlexNet":
        m = tvm.alexnet(weights=tvm.AlexNet_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "VGG16":
        m = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "VGG19":
        m = tvm.vgg19(weights=tvm.VGG19_Weights.DEFAULT)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)
    elif name == "ResNet18":
        m = tvm.resnet18(weights=tvm.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "ResNet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "ResNet101":
        m = tvm.resnet101(weights=tvm.ResNet101_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif name == "DenseNet121":
        m = tvm.densenet121(weights=tvm.DenseNet121_Weights.DEFAULT)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)
    elif name == "EfficientNet-B0":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(name)
    return m

MODEL_NAMES = ["AlexNet", "VGG16", "VGG19", "ResNet18", "ResNet50",
               "ResNet101", "DenseNet121", "EfficientNet-B0"]

## 6. Train / evaluate helpers

In [ ]:
import time
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
import torch.nn.functional as F

def train_one_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    best_val_acc, best_state = 0.0, None

    for epoch in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                preds = model(x).argmax(1)
                correct += (preds == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        print(f"  epoch {epoch+1}/{epochs} - val_acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model

def evaluate_model(model, loader, num_classes=NUM_CLASSES):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            logits = model(x)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(1)
            all_preds.extend(preds)
            all_labels.extend(y.numpy())
            all_probs.extend(probs)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except ValueError:
        auc = float("nan")

    return {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "AUC (%)": round(auc * 100, 2),
    }

## 7. Table 1 — train & evaluate every model

In [ ]:
import pandas as pd

table1_rows = []
trained_models = {}   # name -> trained model, kept on CPU to save GPU memory

for name in MODEL_NAMES:
    print(f"\n=== Training {name} ===")
    model = build_model(name)
    model = train_one_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR)
    metrics = evaluate_model(model, test_loader)
    metrics["Model"] = name
    table1_rows.append(metrics)
    trained_models[name] = model.cpu()   # move to CPU, reload to GPU only when needed
    torch.cuda.empty_cache()

table1_df = pd.DataFrame(table1_rows)[
    ["Model", "Accuracy (%)", "Precision (%)", "Recall (%)", "F1-Score (%)", "AUC (%)"]
]
print("\nTABLE 1 — Transfer Learning Models")
print(table1_df.to_string(index=False))
table1_df.to_csv("/content/table1_models.csv", index=False)
table1_df

## 8. Table 2 — deep-feature extraction + classical classifiers
Pick which backbone supplies the "Deep Features" — `ResNet50` is a solid default.

In [ ]:
FEATURE_BACKBONE = "ResNet50"

def get_feature_extractor(model, name):
    """Strip the classification head so forward() returns the penultimate
    feature vector instead of class logits."""
    model = model.to(DEVICE).eval()
    if name.startswith("VGG") or name == "AlexNet":
        model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
        return model
    if name.startswith("ResNet"):
        modules = list(model.children())[:-1]  # drop fc
        return nn.Sequential(*modules, nn.Flatten())
    if name == "DenseNet121":
        feat = model.features
        return nn.Sequential(feat, nn.ReLU(inplace=True),
                              nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())
    if name == "EfficientNet-B0":
        model.classifier = nn.Sequential(*list(model.classifier.children())[:-1])
        return model
    raise ValueError(name)

def extract_features(extractor, loader):
    feats, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            f = extractor(x)
            f = torch.flatten(f, 1)
            feats.append(f.cpu().numpy())
            labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)

backbone_model = trained_models[FEATURE_BACKBONE]
extractor = get_feature_extractor(backbone_model, FEATURE_BACKBONE)

print("Extracting deep features with", FEATURE_BACKBONE, "...")
X_train, y_train = extract_features(extractor, train_loader)
X_val,   y_val   = extract_features(extractor, val_loader)
X_test,  y_test  = extract_features(extractor, test_loader)

# fold val into train for the classical classifiers (they don't need a val loop)
X_train_full = np.concatenate([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train_full)
X_train_full = scaler.transform(X_train_full)
X_test_s = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000, n_jobs=-1),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=7),
    "Linear SVM": SVC(kernel="linear", probability=True),
    "RBF-SVM": SVC(kernel="rbf", probability=True),
    "XGBoost": XGBClassifier(eval_metric="mlogloss", n_jobs=-1),
}

table2_rows = []
for clf_name, clf in classifiers.items():
    print(f"Training {clf_name} on deep features...")
    clf.fit(X_train_full, y_train_full)
    preds = clf.predict(X_test_s)
    probs = clf.predict_proba(X_test_s) if hasattr(clf, "predict_proba") else None

    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average="macro", zero_division=0)
    rec = recall_score(y_test, preds, average="macro", zero_division=0)
    f1 = f1_score(y_test, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_test, probs, multi_class="ovr", average="macro") if probs is not None else float("nan")
    except ValueError:
        auc = float("nan")

    table2_rows.append({
        "Feature Extractor": "Deep Features",
        "Classifier": clf_name,
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "AUC (%)": round(auc * 100, 2),
    })

table2_df = pd.DataFrame(table2_rows)
print("\nTABLE 2 — Classifiers on Deep Features (backbone:", FEATURE_BACKBONE, ")")
print(table2_df.to_string(index=False))
table2_df.to_csv("/content/table2_classifiers.csv", index=False)
table2_df

## 9. Table 3 — computational efficiency

In [ ]:
from thop import profile

def count_params_millions(model):
    return sum(p.numel() for p in model.parameters()) / 1e6

def model_size_mb(model):
    tmp_path = "/content/_tmp_model.pt"
    torch.save(model.state_dict(), tmp_path)
    size = os.path.getsize(tmp_path) / (1024 * 1024)
    os.remove(tmp_path)
    return size

def flops_g(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE)):
    model = model.to(DEVICE).eval()
    dummy = torch.randn(*input_size).to(DEVICE)
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    return macs * 2 / 1e9  # FLOPs ~= 2 * MACs, in GFLOPs

def inference_time_ms(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), n_runs=50):
    model = model.to(DEVICE).eval()
    dummy = torch.randn(*input_size).to(DEVICE)
    with torch.no_grad():
        for _ in range(10):
            model(dummy)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            model(dummy)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    end = time.time()
    return (end - start) / n_runs * 1000  # ms per image

table3_rows = []
acc_lookup = dict(zip(table1_df["Model"], table1_df["Accuracy (%)"]))

for name in MODEL_NAMES:
    print(f"Profiling {name}...")
    model = trained_models[name]
    params_m = count_params_millions(model)
    size_mb = model_size_mb(model)
    gflops = flops_g(model)
    infer_ms = inference_time_ms(model)

    table3_rows.append({
        "Model": name,
        "Parameters (M)": round(params_m, 2),
        "Model Size (MB)": round(size_mb, 2),
        "FLOPs (G)": round(gflops, 2),
        "Inference Time (ms)": round(infer_ms, 2),
        "Accuracy (%)": acc_lookup[name],
    })
    model.cpu()
    torch.cuda.empty_cache()

table3_df = pd.DataFrame(table3_rows)
print("\nTABLE 3 — Computational Efficiency")
print(table3_df.to_string(index=False))
table3_df.to_csv("/content/table3_efficiency.csv", index=False)
table3_df

## 10. Export all three tables to one Excel workbook

In [ ]:
with pd.ExcelWriter("/content/skin_cancer_results.xlsx") as writer:
    table1_df.to_excel(writer, sheet_name="Table1_Models", index=False)
    table2_df.to_excel(writer, sheet_name="Table2_Classifiers", index=False)
    table3_df.to_excel(writer, sheet_name="Table3_Efficiency", index=False)

from google.colab import files as colab_files
colab_files.download("/content/skin_cancer_results.xlsx")